<a href="https://colab.research.google.com/github/chandupradeep18/Building-with-the-Claude-API/blob/main/Claude_With_Tool_Calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install anthropic openai

In [ ]:
from anthropic import Anthropic
from anthropic.types import ToolParam,Message
from openai import OpenAI
import requests,json,csv,os
from google.colab import userdata
from datetime import datetime,timedelta
nebius = OpenAI(base_url="https://api.tokenfactory.nebius.com/v1/",api_key=userdata.get("NEBIUS"))
nvidia = OpenAI(base_url="https://integrate.api.nvidia.com/v1",api_key=userdata.get('NVIDIA'))
claude = Anthropic(api_key=userdata.get('CLAUDE'))
ollama = OpenAI(base_url="https://ollama.com/v1",api_key=userdata.get('OLLAMA'))
nebius_model={'api':'nebius','model':"nvidia/Nemotron-3.5-Super-VL"}
nvidia_model={'api':'nvidia','model':"nvidia/nemotron-3.5-super-vl-120b-a12b"}
claude_model={'api':'claude','model':"claude-haiku-4-5"}
ollama_gemma4_model={'api':'ollama','model':"gemma4:31b"}

In [ ]:
def add_user_message(messages, message):
    messages.append({"role": "user", "content": message.content if isinstance(message,Message) else message})
def add_assistant_message(messages, message):
    messages.append({"role": "assistant", "content": message.content if isinstance(message,Message) else message})
def collect_chat_history(response):
  file_exist=os.path.exists('/content/drive/MyDrive/Response/chat_history.csv')
  with open('/content/drive/MyDrive/Response/chat_history.csv','a',newline='') as save_response:
    writer=csv.DictWriter(save_response,fieldnames=response.keys())
    if not file_exist:
      writer.writeheader()
    writer.writerow(response)
def openai_client(client,messages,model,system=None,temperature=0.5,max_tokens=1800):
  payload = {"model": model['model'],"messages": messages,"temperature": temperature,"max_tokens": max_tokens,"stream": False,}
  if system:
    messages.append({"role": "system", "content": system})
  response = client.chat.completions.create(**payload)
  message = response.choices[0].message
  usage = response.usage
  details={ "Client": "OpenAi",'api':model['api'],'model':model['model'],"Time": datetime.now().isoformat(),"id": response.id,"role": message.role,"stop_reason": response.choices[0].finish_reason,
    "input_tokens": usage.prompt_tokens,"output_tokens": usage.completion_tokens,"total_tokens": usage.total_tokens,"user": messages[-1]["content"] ,"assistant": message.content}
  collect_chat_history(details)
  return response.choices[0].message.content
def claude_client(client,messages,model,system=None,max_tokens=1800,stop_sequences=[],tool=None):
  params = {"model": model['model'],"max_tokens": max_tokens,"messages": messages,"stop_sequences": stop_sequences}
  if tool:
    params["tools"] = tool
  if system:
    params["system"] = system
  response = client.messages.create(**params)
  details={'Client':'Claude','api':model['api'],'model':model['model'],'Time':datetime.now().isoformat(),'id':response.id,'role':response.role,'stop_reason':response.stop_reason,
           'input_tokens':response.usage.input_tokens,'output_tokens':response.usage.output_tokens,"total_tokens": response.usage.output_tokens+response.usage.input_tokens,
           'user':messages[-1]['content'],'assistant':response.content}
  collect_chat_history(details)
  return response

In [ ]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
  if not date_format:
    raise ValueError("date_format cannot be empty")
  return datetime.now().strftime(date_format)
def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")

def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")

get_current_datetime_schema=ToolParam({
  "name": "get_current_datetime",
  "description": "Returns the current date and time as a formatted string, using the system clock of the machine that runs the tool. Use it whenever the user asks for the current date, time, day, or timestamp, or when you need today's date to reason about deadlines, schedules, or relative dates such as 'tomorrow' or 'next week'. Do not use it for dates in other time zones, for date arithmetic, or for parsing existing dates. The result is in the server's local time and does not include a time zone or UTC offset unless you add one through the format string. The optional date_format parameter controls the output layout. If it is omitted, the tool returns a timestamp like '2026-10-26 20:28:00'. The tool returns an error if date_format is an empty string.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "minLength": 1,
        "default": "%Y-%m-%d %H:%M:%S",
        "description": "Optional. A Python strftime format string that controls how the date and time are rendered. Defaults to '%Y-%m-%d %H:%M:%S' (e.g. '2026-10-26 20:28:00'). Common directives: %Y year, %m month, %d day, %H hour (24h), %I hour (12h), %M minute, %S second, %p AM/PM, %A weekday name, %B month name, %Z time zone name. Must be a non-empty string; omit the parameter instead of passing an empty value."
      }
    },
    "required": ["date_format"],
  },
  "input_examples": [
    { "date_format": "%Y-%m-%d" },
    { "date_format": "%Y-%m-%d %H:%M:%S" }
  ]
})
add_duration_to_datetime_schema = ToolParam({
    "name": "add_duration_to_datetime",
    "description": (
        "Adds a duration to a given date or datetime and returns the resulting date and time as a "
        "human-readable string, always in the format 'Weekday, Month DD, YYYY HH:MM:SS AM/PM' "
        "(e.g. 'Wednesday, November 05, 2026 12:00:00 AM'). Use it for date arithmetic, such as finding "
        "the date 10 days from a given date, a deadline 3 weeks after a start date, or a time 2 hours "
        "before an event; pass a negative duration to subtract. Do not use it to find the current date "
        "(use get_current_datetime for that), to calculate the difference between two dates, or to "
        "convert between time zones. The input is parsed with input_format, which must match "
        "datetime_str exactly or the tool returns an error. If the input has no time component, the time "
        "is treated as midnight. Month arithmetic clamps the day to the end of the target month (Jan 31 "
        "plus 1 month gives Feb 28 or 29). Year arithmetic fails for Feb 29 when the target year is not a "
        "leap year. An unsupported unit also returns an error."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": (
                    "The starting date or datetime, written to match input_format. With the default "
                    "format this is a date such as '2026-10-26'. If you pass a datetime such as "
                    "'2026-10-26 14:30:00', you must also set input_format to '%Y-%m-%d %H:%M:%S'."
                ),
            },
            "duration": {
                "type": "integer",
                "default": 0,
                "description": (
                    "Optional. The whole number of units to add. Use a negative number to subtract. "
                    "Defaults to 0, which returns the starting date unchanged."
                ),
            },
            "unit": {
                "type": "string",
                "enum": ["seconds", "minutes", "hours", "days", "weeks", "months", "years"],
                "default": "days",
                "description": (
                    "Optional. The unit of the duration. Defaults to 'days'. Months and years are "
                    "calendar-based; the other units are fixed lengths of time."
                ),
            },
            "input_format": {
                "type": "string",
                "minLength": 1,
                "default": "%Y-%m-%d",
                "description": (
                    "Optional. A Python strptime format string describing how datetime_str is written. "
                    "Defaults to '%Y-%m-%d'. Common directives: %Y year, %m month, %d day, %H hour (24h), "
                    "%I hour (12h), %M minute, %S second, %p AM/PM. This only affects parsing the input; "
                    "the output format is fixed."
                ),
            },
        },
        "required": ["datetime_str"]
    }
})
set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

In [ ]:
def run_tools(response):
  tool_requests = [block for block in response.content if block.type == "tool_use"]
  results=[]
  for tool in tool_requests:
    try:
        if tool.name=='get_current_datetime':
          tool_results=get_current_datetime(**tool.input)
        elif tool.name=='add_duration_to_datetime':
          tool_results=add_duration_to_datetime(**tool.input)
        elif tool.name=='set_reminder':
          tool_results=set_reminder(**tool.input)
        results.append({"type": "tool_result","tool_use_id": tool.id,"content":tool_results,"is_error": False})
    except Exception as e:
      results.append({"type": "tool_result","tool_use_id": tool.id,"content":str(e),"is_error": True})
  return results
def conversation(messages,tools=[], fine_grained=False):
  while True:
    if fine_grained:
       with claude.beta.messages.stream(model=claude_model['model'],messages=messages,max_tokens=1000,
       tools=tools,betas=['fine-grained-tool-streaming-2025-05-14'] if fine_grained else [],
       ) as streams:
         for chunk in streams:
          if chunk.type == "text":
            print(chunk.text, end="")
          if chunk.type == "content_block_start":
            if chunk.content_block.type == "tool_use":
                print(f'\n>>> Tool Call: "{chunk.content_block.name}"')
          if chunk.type == "input_json" and chunk.partial_json:
                print(chunk.partial_json, end="")
         response = streams.get_final_message()
    else:
      response=claude_client(claude,messages,claude_model,max_tokens=1000,tool=tools)
    add_assistant_message(messages,response.content)
    if response.stop_reason!='tool_use':
      break
    tool_results=run_tools(response)
    add_user_message(messages, tool_results)
  return messages

In [ ]:
chat_history=[]
add_user_message(chat_history,'Set a Reminder For my Interview at Google it from 2 weeks 4 days at 1:00 pm from today')
response=conversation(chat_history,tools=[get_current_datetime_schema,add_duration_to_datetime_schema,set_reminder_schema],fine_grained=True)

I'll help you set a reminder for your Google interview. First, let me get today's date and then calculate the date for 2 weeks and 4 days from today.
>>> Tool Call: "get_current_datetime"
{"date_format": "%Y-%m-%d %H:%M:%S"}Now let me calculate the date that is 2 weeks and 4 days from today (which is 18 days total):
>>> Tool Call: "add_duration_to_datetime"
{"datetime_str": "2026-10-29", "duration": 18, "unit": "days"}Perfect! Now let me set the reminder for your Google interview at 1:00 PM on November 16, 2026:
>>> Tool Call: "set_reminder"
{"content": "Interview at Google", "timestamp": "2026-11-16T13:00:00"}----
Setting the following reminder for 2026-11-16T13:00:00:
Interview at Google
----
Great! ✅ I've successfully set a reminder for your **Google Interview** on **Monday, November 16, 2026 at 1:00 PM** (which is 2 weeks and 4 days from today). You'll receive a notification at that time.

In [ ]:
response[-1]['content'][0].text

"Great! ✅ I've successfully set a reminder for your **Google Interview** on **Monday, November 16, 2026 at 1:00 PM** (which is 2 weeks and 4 days from today). You'll receive a notification at that time."